# 03 · Model comparison
**Objectives:** compare the full 15-entry riddle dataset, inspect answer checks and rubrics, and distinguish quality from operational measurements.

The original dataset includes a repeated breath riddle. Keep it to discuss duplicated weighting. Live mode makes **45 response requests and 12 rubric requests** with the default three-model comparison. All runs are fresh; there is no response cache. Model outputs vary and synthetic answers are deliberately identical across models.


In [ ]:
from pathlib import Path
import json
import os
import sys

# Works from labs/boto3 or its notebooks directory, including a fresh clone.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").is_file() and (p / "data/synthetic.json").is_file())
sys.path.insert(0, str(ROOT / "src"))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env", override=False)
from ai_eval_labs.core import Bedrock, DATA, Result, export_results, generate, load_cases
from ai_eval_labs.metrics import Judge, check, contains

LIVE = False  # Set True only after reviewing AWS model access, prices, and request counts.
api = Bedrock(live=LIVE)
judge = Judge(api)
synthetic = json.loads((DATA / "synthetic.json").read_text())
print("LIVE: paid Bedrock calls" if LIVE else "OFFLINE: synthetic teaching fixtures; no AWS calls")


In [ ]:
from ai_eval_labs.lessons import evaluate_riddle
cases = load_cases("riddles")
models = [x.strip() for x in os.getenv("COMPARISON_MODEL_IDS", "openai.gpt-oss-120b-1:0,us.openai.gpt-6-luna,us.openai.gpt-6.1-sol").split(",") if x.strip()]
assert models, "Configure at least one comparison model"
INPUT_USD_PER_MILLION = None  # Optional {model_id: rate}; supply current rates yourself.
OUTPUT_USD_PER_MILLION = None
len(cases), models


## Worked example: explicit evaluation
Content checks mirror the original cases. Rubrics return a validated JSON boolean and reason; malformed grades become errors. The inherited limits are 60 seconds and $0.15 per response. They do not stop requests or account for grader charges.

Cost = `(input tokens × input rate + output tokens × output rate) / 1,000,000`. Without both model-specific rates, cost is unknown. Synthetic timing/cost checks are unavailable, rather than passing.


In [ ]:
results = []
for model in models:
    for case, answer in zip(cases, synthetic["riddles"]):
        result = generate(api, case["id"], f"Solve this riddle: {case['riddle']}", answer, model)
        evaluate_riddle(result, case, judge,
            input_rate=(INPUT_USD_PER_MILLION or {}).get(model),
            output_rate=(OUTPUT_USD_PER_MILLION or {}).get(model))
        results.append(result)
[(r.scenario_id, r.model_id, r.response, r.assertions, r.errors) for r in results[:3]]


In [ ]:
# Aggregate only evaluated checks; report unavailable checks and errors separately.
for model in models:
    rows = [r for r in results if r.model_id == model]
    checks = [a for r in rows for a in r.assertions if a["passed"] is not None]
    print(model, {"passed": sum(a["passed"] for a in checks), "evaluated": len(checks),
                  "unavailable": sum(a["passed"] is None for r in rows for a in r.assertions),
                  "errors": sum(len(r.errors) for r in rows)})
export_results(results, ROOT / "results/03_comparison")


## Exercise
Change one riddle's rubric to accept a defensible alternative answer, then compare it with an exact substring check. Remove the repeated case in an in-memory copy and explain how its weight changes the aggregate. For live work, repeat a small subset before drawing conclusions about model rankings.

**Hint:** use the per-scenario assertions and grader reasons rather than one overall pass rate.

**Checkpoints:** Does one model's passing count imply superior reasoning? Which requests are billed beyond generation? Why are thresholds not budgets? Where might judge preference influence the result?

Next: [RAG evaluation](04_rag_evaluation.ipynb).
